In [7]:
import requests
import json
from datetime import datetime, date
import time
import notebookutils
import pandas_market_calendars as mcal
import os

StatementMeta(, 6f955622-a76e-432c-8edf-495ed129a1aa, 11, Finished, Available, Finished, False)

# 0. Check if the market open

In [8]:
def is_market_open():
    # Load the NYSE calendar
    nyse = mcal.get_calendar('NYSE')
    
    # Check if today is a trading day
    today = date.today()
    schedule = nyse.schedule(start_date=today, end_date=today)
    
    # If the schedule is empty, the market is closed
    return not schedule.empty

# --- THE TRIGGER ---
if not is_market_open():
    print(f"🚫 Market is closed today ({date.today()}). Exiting gracefully.")
    notebookutils.notebook.exit("Success: Market is closed, skipping extraction.")

print(f"✅ Market is open today ({date.today()}). Proceeding with extraction.")

StatementMeta(, 6f955622-a76e-432c-8edf-495ed129a1aa, 12, Finished, Available, Finished, False)

✅ Market is open today (2026-06-17). Proceeding with extraction.


# 1. Setup variables

In [9]:
#Retrieve the Variable Library
vl = notebookutils.variableLibrary.getLibrary("Var")

#Extract the API Key
api_key = vl.getVariable("API_KEY_ALPHA_VANTAGE")

base_url = "https://www.alphavantage.co/query"
today_str = datetime.now().strftime("%Y-%m-%d")

# Define the OneLake path (Files is the Bronze layer)
relative_folder_path = f"Files/bronze/{today_str}"
absolute_folder_path = f"file:/lakehouse/default/Files/bronze/{today_str}"

StatementMeta(, 6f955622-a76e-432c-8edf-495ed129a1aa, 13, Finished, Available, Finished, False)

# 2. Extract Most Active Stocks

In [10]:
print("\nExtracting Most Active Stocks...")
object_name = f"{absolute_folder_path}/most_active_stocks.json"

# GRANULAR SAFEGUARD: Check if the file already exists
if os.path.exists(object_name):
    print(f"⏭️ Most active stocks file already exists. Skipping API call.")
    
    # CRITICAL: We must read the existing file to find out what the top 3 stocks were!
    with open(object_name, 'r', encoding='utf-8') as f:
        most_active_stocks = json.load(f)
        
else:
    
    notebookutils.fs.mkdirs(relative_folder_path)
    notebookutils.fs.mkdirs(f"{relative_folder_path}/news")
    notebookutils.fs.mkdirs(f"{relative_folder_path}/business_info")
    print(f"Created directory structure for {today_str}")
    
    # File does not exist, so we burn 1 API call to get the data
    response = requests.get(
        base_url,
        params={'function': 'TOP_GAINERS_LOSERS', 'apikey': api_key},
        timeout=10
    )
    response.raise_for_status()
    data = response.json()
    most_active_stocks = data.get('most_actively_traded', [])
    
    # Slice the list to keep only the top 3
    most_active_stocks = most_active_stocks[:3]

    # Save the fresh data to OneLake
    notebookutils.fs.put(object_name, json.dumps(most_active_stocks, ensure_ascii=False), True)
    print(f"Stored: {object_name}")

# Now we safely extract the top 3, whether they came from the API or the existing file!
top3_stocks = [stock['ticker'] for stock in most_active_stocks[:3]]
print(f"Identified Top 3 stocks to process next: {top3_stocks}")

StatementMeta(, 6f955622-a76e-432c-8edf-495ed129a1aa, 14, Finished, Available, Finished, False)


Extracting Most Active Stocks...
Created directory structure for 2026-06-17
Stored: file:/lakehouse/default/Files/bronze/2026-06-17/most_active_stocks.json
Identified Top 3 stocks to process next: ['ADTX', 'GDC', 'SOXS']


# 3.Extract Top 3 Stocks News & Biz info

In [11]:
print(f"Starting detailed extraction for top 3 stocks: {top3_stocks}")

# --- 1. Extract News Sentiment ---
print("\nExtracting News Sentiment...")
for index, symbol in enumerate(top3_stocks):
    news_object_name = f"{absolute_folder_path}/news/{symbol}_stocks_news.json"
    
    # GRANULAR SAFEGUARD: Skip this specific API call if the file exists
    if os.path.exists(news_object_name):
        print(f"⏭️ News for {symbol} already exists. Skipping.")
        continue 
        
    response = requests.get(
        base_url,
        params={'function': 'NEWS_SENTIMENT', 'tickers': symbol, 'apikey': api_key},
        timeout=10
    )
    notebookutils.fs.put(news_object_name, json.dumps(response.json(), ensure_ascii=False), True)
    print(f"Stored news for {symbol}")
    time.sleep(2) # Respect API rate limits


# --- 2. Extract Business Info ---
print("\nExtracting Business Info...")
for index, symbol in enumerate(top3_stocks):
    biz_object_name = f"{absolute_folder_path}/business_info/{symbol}_stocks_business_info.json"
    
    # GRANULAR SAFEGUARD: Skip this specific API call if the file exists
    if os.path.exists(biz_object_name):
        print(f"⏭️ Business info for {symbol} already exists. Skipping.")
        continue

    response = requests.get(
        base_url,
        params={'function': 'OVERVIEW', 'symbol': symbol, 'apikey': api_key},
        timeout=10
    )
    notebookutils.fs.put(biz_object_name, json.dumps(response.json(), ensure_ascii=False), True)
    print(f"Stored business info for {symbol}")
    time.sleep(2)

# print("\nAll Bronze layer extractions complete!")

StatementMeta(, 6f955622-a76e-432c-8edf-495ed129a1aa, 15, Finished, Available, Finished, False)

Starting detailed extraction for top 3 stocks: ['ADTX', 'GDC', 'SOXS']

Extracting News Sentiment...
Stored news for ADTX
Stored news for GDC
Stored news for SOXS

Extracting Business Info...
Stored business info for ADTX
Stored business info for GDC
Stored business info for SOXS


In [12]:
# --- 3. Extract Daily Price Data ---
print("\nExtracting Daily Price Data...")

# Ensure the 'price' directory exists in today's Bronze folder
notebookutils.fs.mkdirs(f"{relative_folder_path}/price")

for index, symbol in enumerate(top3_stocks):
    price_object_name = f"{absolute_folder_path}/price/{symbol}_stocks_price.json"
    
    # GRANULAR SAFEGUARD: Skip this specific API call if the file already exists
    if os.path.exists(price_object_name):
        print(f"⏭️ Price data for {symbol} already exists. Skipping.")
        continue 
        
    # Call the Alpha Vantage API for daily time series
    response = requests.get(
        base_url,
        params={'function': 'TIME_SERIES_DAILY', 'symbol': symbol, 'apikey': api_key},
        timeout=10
    )
    
    # Catch any HTTP errors before saving
    response.raise_for_status()
    
    # Save the JSON payload to the OneLake Bronze layer
    notebookutils.fs.put(price_object_name, json.dumps(response.json(), ensure_ascii=False), True)
    print(f"Stored price data for {symbol}")
    
    # Respect the Alpha Vantage free-tier rate limits
    time.sleep(2) 

print("\n✅ All Bronze layer extractions (including price) are completely finished!")

StatementMeta(, 6f955622-a76e-432c-8edf-495ed129a1aa, 16, Finished, Available, Finished, False)


Extracting Daily Price Data...
Stored price data for ADTX
Stored price data for GDC
Stored price data for SOXS

✅ All Bronze layer extractions (including price) are completely finished!
